In [3]:
!pip install -q transformers torch pandas


# Laboratorio 4

[Link Repositorio](https://github.com/donmatthiuz/NLP/tree/lab4)

## Imports

In [15]:
import pandas as pd
from transformers import pipeline
import json




## Dataset

El dataset Dataset for Sentiment Analysis in Spanish, creado por Francisco José Ramírez Vicente. Contiene reseñas obtenidas de Google Maps y Tripadvisor, con las columnas puntuación, review y sentimiento. Incluye opiniones positivas, negativas, neutrales/mixtas, negaciones y menciones de personas y lugares.

In [5]:

url = "https://raw.githubusercontent.com/fjramirezv/sentiment-webscraping/main/dataset_sentiment_analisys.csv"

df = pd.read_csv(
    url,
    sep=";",
    encoding="utf-8-sig",
    engine="python",
    on_bad_lines="skip"
)

# Cambiar nombres de columnas
df = df.rename(columns={
    "puntuación": "puntuacion",
    "review": "reseña"
})

# Eliminar filas vacías
df = df.dropna(subset=["reseña", "sentimiento"])
df = df[df["reseña"].str.strip() != ""]

# Mantener ejemplos de cada sentimiento
obligatorias = pd.concat([
    df[df["sentimiento"] == "positivo"].head(10),
    df[df["sentimiento"] == "negativo"].head(5),
    df[df["sentimiento"] == "neutral"].head(5)
])

restantes = df.drop(obligatorias.index)
cantidad_faltante = 30 - len(obligatorias)

dataset_30 = pd.concat([
    obligatorias,
    restantes.head(cantidad_faltante)
]).head(30).reset_index(drop=True)

# Considerar las opiniones neutrales con aspectos buenos y malos como mixtas
dataset_30["tipo"] = dataset_30["sentimiento"].replace({
    "neutral": "mixto"
})

# Detectar posibles negaciones
dataset_30["tiene_negacion"] = dataset_30["reseña"].str.contains(
    r"\b(no|nunca|jamás|tampoco|ni|sin)\b",
    case=False,
    regex=True
)

display(dataset_30)

/tmp/ipykernel_2372/562147463.py:42: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  dataset_30["tiene_negacion"] = dataset_30["reseña"].str.contains(


,puntuacion,reseña,sentimiento,tipo,tiene_negacion
0,40.0,Dispone de espacios acogedores con unas mampar...,positivo,positivo,False
1,40.0,"Tanto la comida como los pintxos son buenos, l...",positivo,positivo,False
2,50.0,Hemos comido estupendamente en la terraza con ...,positivo,positivo,False
3,50.0,Excelente comida en una de las mejores terraza...,positivo,positivo,False
4,40.0,"Para comer o cenar algo esta bien, lo malo es ...",positivo,positivo,False
5,40.0,un sitio muy agradable para tomar un café unas...,positivo,positivo,False
6,50.0,Conocí este sitio a través de internet y me pa...,positivo,positivo,False
7,50.0,Tras pedirme un café y le pedí consejo al cama...,positivo,positivo,False
8,50.0,Sitio agradable y familiar . El trato por el p...,positivo,positivo,False
9,50.0,El bully nunca falla como escenario para nuest...,positivo,positivo,True


## 5. Parte A: Sentimiento

In [7]:

# Modelo de sentimiento entrenado para textos en español
sentiment = pipeline(
    "sentiment-analysis",
    model="pysentimiento/robertuito-sentiment-analysis"
)

# Convertir la columna de reseñas en una lista
reseñas = dataset_30["reseña"].fillna("").astype(str).tolist()

# Procesar las reseñas en lotes de 8
resultados = sentiment(
    reseñas,
    batch_size=8,
    truncation=True
)

# Traducir las etiquetas del modelo
etiquetas = {
    "POS": "positivo",
    "NEG": "negativo",
    "NEU": "neutral"
}

# Guardar los resultados por reseña
df_resultados = pd.DataFrame([
    {
        "identificador": indice + 1,
        "texto_original": texto,
        "etiqueta_sentimiento": etiquetas.get(
            resultado["label"],
            resultado["label"]
        ),
        "score_confianza": round(float(resultado["score"]), 4)
    }
    for indice, (texto, resultado) in enumerate(zip(reseñas, resultados))
])

display(df_resultados)


# Guaradarlos en un csv
df_resultados.to_csv(
    "resultados_sentimiento.csv",
    index=False,
    encoding="utf-8-sig"
)

config.json:   0%|          | 0.00/925 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  435MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/384 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.31M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/167 [00:00<?, ?B/s]

,identificador,texto_original,etiqueta_sentimiento,score_confianza
0,1,Dispone de espacios acogedores con unas mampar...,positivo,0.9288
1,2,"Tanto la comida como los pintxos son buenos, l...",positivo,0.9817
2,3,Hemos comido estupendamente en la terraza con ...,positivo,0.9789
3,4,Excelente comida en una de las mejores terraza...,positivo,0.9791
4,5,"Para comer o cenar algo esta bien, lo malo es ...",negativo,0.7070
5,6,un sitio muy agradable para tomar un café unas...,positivo,0.9508
6,7,Conocí este sitio a través de internet y me pa...,positivo,0.9429
7,8,Tras pedirme un café y le pedí consejo al cama...,positivo,0.9300
8,9,Sitio agradable y familiar . El trato por el p...,positivo,0.9735
9,10,El bully nunca falla como escenario para nuest...,positivo,0.9764


## 6. Parte B: NER


In [12]:

## Pipeline
ner = pipeline(
    "token-classification",
    model="Davlan/bert-base-multilingual-cased-ner-hrl",
    aggregation_strategy="simple"
)

# Utiliza la misma lista de reseñas de la Parte A
reseñas = dataset_30["reseña"].fillna("").astype(str).tolist()

# Procesar las reseñas en lote
resultados_ner = ner(
    reseñas,
    batch_size=8,
)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

In [13]:
entidades_extraidas = []

for identificador, (texto, entidades) in enumerate(
    zip(reseñas, resultados_ner),
    start=1
):
    for entidad in entidades:

        # La clave depende del modelo
        tipo = entidad.get(
            "entity_group",
            entidad.get("entity", "DESCONOCIDO")
        )

        entidades_extraidas.append({
            "identificador_reseña": identificador,
            "texto_original": texto,
            "texto_entidad": entidad.get("word", "").strip(),
            "tipo_entidad": tipo,
            "score": round(float(entidad.get("score", 0)), 4)
        })

df_entidades = pd.DataFrame(
    entidades_extraidas,
    columns=[
        "identificador_reseña",
        "texto_original",
        "texto_entidad",
        "tipo_entidad",
        "score"
    ]
)

display(df_entidades)

,identificador_reseña,texto_original,texto_entidad,tipo_entidad,score
0,4,Excelente comida en una de las mejores terraza...,Donosti,LOC,0.9993
1,20,"Es un sitio genial, con camareros muy atentos ...",Claudia,PER,0.9909
2,22,"Recomendable si quieres sentirte como en casa,...",Ibai,PER,0.9945
3,22,"Recomendable si quieres sentirte como en casa,...",Lander,PER,0.9992
4,22,"Recomendable si quieres sentirte como en casa,...",Iker,PER,0.9993
5,23,"Comer en Donosti, al sol en la terraza y en No...",Donosti,LOC,0.9987
6,23,"Comer en Donosti, al sol en la terraza y en No...",Nochebuena,LOC,0.9474
7,23,"Comer en Donosti, al sol en la terraza y en No...",Lander,PER,0.9987
8,23,"Comer en Donosti, al sol en la terraza y en No...",Ibai,PER,0.9979
9,24,Servicio rápido y de calidad. Perfecta atenció...,Ibai,PER,0.9427


In [14]:

## GUardarlo
df_entidades.to_csv(
    "resultados_ner.csv",
    index=False,
    encoding="utf-8-sig"
)

## 7. Parte C: Flujo Integrado


In [16]:

mapa_sentimiento = {
    "POS": "POSITIVE",
    "NEG": "NEGATIVE",
    "NEU": "NEUTRAL"
}

resultados_combinados = []

for review_id, (texto, sentimiento, entidades) in enumerate(
    zip(reseñas, resultados, resultados_ner),
    start=1
):
    etiqueta_original = sentimiento.get("label", "UNKNOWN")

    entidades_formateadas = []

    for entidad in entidades:
        # La etiqueta puede variar según el modelo
        etiqueta_entidad = entidad.get(
            "entity_group",
            entidad.get("entity", "UNKNOWN")
        )

        entidades_formateadas.append({
            "label": etiqueta_entidad,
            "text": entidad.get("word", "").strip(),
            "score": round(float(entidad.get("score", 0)), 4)
        })

    resultados_combinados.append({
        "review_id": review_id,
        "text": texto,
        "sentiment": mapa_sentimiento.get(
            etiqueta_original,
            etiqueta_original
        ),
        "sentiment_score": round(
            float(sentimiento.get("score", 0)),
            4
        ),
        "entities": entidades_formateadas
    })

In [21]:
## Json del primer registro

primera_con_entidades = next(
    (
        resultado
        for resultado in resultados_combinados
        if len(resultado["entities"]) > 0
    ),
    None
)

if primera_con_entidades:
    print(
        json.dumps(
            primera_con_entidades,
            ensure_ascii=False,
            indent=4
        )
    )
else:
    print("No se encontraron reseñas con entidades.")

{
    "review_id": 4,
    "text": "Excelente comida en una de las mejores terrazas de Donosti. Atención de 10. Una maravillosa opción en un lugar tranquiloMás",
    "sentiment": "POSITIVE",
    "sentiment_score": 0.9791,
    "entities": [
        {
            "label": "LOC",
            "text": "Donosti",
            "score": 0.9993
        }
    ]
}
